In [8]:
import pandas as pd
Products=pd.read_excel("Data.xlsx")
Products.columns = Products.columns.str.strip()
print(Products.columns.tolist())
Products

['Product', 'Profit', 'Machine hours', 'Labour hours', 'Material units', 'Fixed Setup Cost']


,Product,Profit,Machine hours,Labour hours,Material units,Fixed Setup Cost
0,A,50,2,3,2,800
1,B,60,3,2,3,1000
2,C,40,1,2,1,500
3,D,80,4,3,4,1200


In [9]:
import pandas as pd
R=pd.read_excel("R.xlsx")
R

,Machine capacity 300 hours
0,Labour capacity 250 hours
1,Material capacity 280 units


In [10]:
import pyomo.environ as pyo
!apt-get install -y glpk-utils
model=pyo.ConcreteModel()
#Sets
model.P=pyo.Set(initialize=Products["Product"].tolist())
M=1000
print(list(model.P))
print(Products.columns.tolist())
#Parameters
model.Profit=pyo.Param(model.P,initialize=Products.set_index("Product")["Profit"].to_dict())
model.Machine_hours=pyo.Param(model.P,initialize=Products.set_index("Product")["Machine hours"].to_dict())
model.Labour_hours=pyo.Param(model.P,initialize=Products.set_index("Product")["Labour hours"].to_dict())
model.Material_units=pyo.Param(model.P,initialize=Products.set_index("Product")["Material units"].to_dict())
model.Fixed_Setup_Cost=pyo.Param(model.P,initialize=Products.set_index("Product")["Fixed Setup Cost"].to_dict())

#Decision Variable
model.x=pyo.Var(model.P,within=pyo.NonNegativeReals)
model.y=pyo.Var(model.P,within=pyo.Binary)

#Constraint
def Labour_capacity_Constraint(model,p):
  return sum(model.Labour_hours[p]* model.x[p] for p in model.P)<=250
model.Labour_capacity=pyo.Constraint(model.P,rule=Labour_capacity_Constraint)

#Machine capacity Constraint
def Machine_capacity_Constraint(model,p):
  return sum(model.Machine_hours[p]* model.x[p] for p in model.P)<=300
model.Machine_capacity=pyo.Constraint(model.P,rule=Machine_capacity_Constraint)

def Material_capacity_Constraint(model,p):
  return sum(model.Material_units[p]* model.x[p] for p in model.P)<=280
model.Material_capacity=pyo.Constraint(model.P,rule=Material_capacity_Constraint)


def Linking_Constraint(model,p):
  return model.x[p]<=M * model.y[p]
model.Linking_Constraint=pyo.Constraint(model.P,rule=Linking_Constraint)

def Objective_rule(model):
    return sum(
        model.Profit[p] * model.x[p]
        for p in model.P
    ) - sum(
        model.Fixed_Setup_Cost[p] * model.y[p]
        for p in model.P
    )

model.Objective = pyo.Objective(
    rule=Objective_rule,
    sense=pyo.maximize
)

#Solver
solver = pyo.SolverFactory(
    "glpk",
    executable="/usr/bin/glpsol"
)
results = solver.solve(model)
print(results.solver.status)
print(results.solver.termination_condition)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
glpk-utils is already the newest version (5.0-1build2).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.
['A', 'B', 'C', 'D']
['Product', 'Profit', 'Machine hours', 'Labour hours', 'Material units', 'Fixed Setup Cost']
ok
optimal


In [15]:
print("Optimal Objective:",pyo.value(model.Objective))

Optimal Objective: 5050.0


In [19]:
result_table=pd.DataFrame({
    "Product":[p for p in model.P],
    "Profit":[pyo.value(model.Profit[p]) for p in model.P],
    "Production":[pyo.value(model.x[p]) for p in model.P],
    "Selected":[pyo.value(model.y[p]) for p in model.P],

})
print(result_table)

  Product  Profit  Production  Selected
0       A      50         0.0       0.0
1       B      60        77.5       1.0
2       C      40        47.5       1.0
3       D      80         0.0       0.0


In [35]:
used_labour_hours = sum(
    pyo.value(model.Labour_hours[p]) * pyo.value(model.x[p])
    for p in model.P
)
used_Machine_hours= sum( pyo.value(model.Machine_hours[p])*pyo.value(model.x[p]) for p in model.P)
used_Material_units= sum( pyo.value(model.Material_units[p])*pyo.value(model.x[p]) for p in model.P)
remaining_Labour_hours=250-used_labour_hours
remaining_Machine_hours=300-used_Machine_hours
remaining_Material_units=280-used_Material_units
remaining_table=pd.DataFrame({
    "Resources":["Labour","Machine","Material"],
    "Capacity":[250,300,280],
    "Used":[used_labour_hours,used_Machine_hours,used_Material_units],
    "Remaining":[remaining_Labour_hours,remaining_Machine_hours,remaining_Material_units]
})
print(remaining_table)



  Resources  Capacity   Used  Remaining
0    Labour       250  250.0        0.0
1   Machine       300  280.0       20.0
2  Material       280  280.0        0.0


In [39]:
bottleneck_table = pd.DataFrame({
    "Resource": ["Labour", "Machine", "Material"],
    "Capacity": [250, 300, 280],
    "Used": [
        used_labour_hours,
        used_Machine_hours,
        used_Material_units
    ],
    "Remaining": [
        remaining_Labour_hours,
        remaining_Machine_hours,
        remaining_Material_units
    ]
})

print(bottleneck_table)

#how much of these two resources each product requires.
for p in model.P:
    print(
        p,
        "Profit:", pyo.value(model.Profit[p]),
        "Labour:", pyo.value(model.Labour_hours[p]),
        "Material:", pyo.value(model.Material_units[p]),
        "Setup:", pyo.value(model.Fixed_Setup_Cost[p])
    )

   Resource  Capacity   Used  Remaining
0    Labour       250  250.0        0.0
1   Machine       300  280.0       20.0
2  Material       280  280.0        0.0
A Profit: 50 Labour: 3 Material: 2 Setup: 800
B Profit: 60 Labour: 2 Material: 3 Setup: 1000
C Profit: 40 Labour: 2 Material: 1 Setup: 500
D Profit: 80 Labour: 3 Material: 4 Setup: 1200


In [40]:
for p in model.P:
    print(
        p,
        "Production =", pyo.value(model.x[p]),
        "Machine used =", pyo.value(model.Machine_hours[p]) * pyo.value(model.x[p]),
        "Labour used =", pyo.value(model.Labour_hours[p]) * pyo.value(model.x[p]),
        "Material used =", pyo.value(model.Material_units[p]) * pyo.value(model.x[p]),
        "Setup cost =", pyo.value(model.Fixed_Setup_Cost[p]) * pyo.value(model.y[p])
    )

A Production = 0.0 Machine used = 0.0 Labour used = 0.0 Material used = 0.0 Setup cost = 0.0
B Production = 77.5 Machine used = 232.5 Labour used = 155.0 Material used = 232.5 Setup cost = 1000.0
C Production = 47.5 Machine used = 47.5 Labour used = 95.0 Material used = 47.5 Setup cost = 500.0
D Production = 0.0 Machine used = 0.0 Labour used = 0.0 Material used = 0.0 Setup cost = 0.0
